In [17]:
# ============================================================
# INVESTIGADOR COMPLETO CON LANGGRAPH
# ============================================================
# ------------------------------------------------------------
# IMPORTS INVESTIGADOR
# ------------------------------------------------------------

from typing import TypedDict, List, Dict, Annotated
from dotenv import load_dotenv
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_core.tools import tool
from langchain_cohere import ChatCohere
from langchain_community.utilities import GoogleSerperAPIWrapper
import os




In [18]:
load_dotenv(override=True)

True

In [19]:
serper = GoogleSerperAPIWrapper(
    serper_api_key=os.getenv("SERPER_API_KEY")
)


In [20]:

load_dotenv(override=True)

key = os.getenv("SERPER_API_KEY")
print("SERPER_API_KEY cargada:", bool(key), key[:5] + "..." if key else None)

serper = GoogleSerperAPIWrapper(serper_api_key=key)
print(serper.run("Coldplay upcoming concerts Europe"))


SERPER_API_KEY cargada: True e0555...
There are no dates currently scheduled. Sign up to be the first to hear about upcoming shows. READ OUR LATEST TOUR EMISSIONS UPDATE. Buy Coldplay tickets from the official Ticketmaster.com site. Find Coldplay tour schedule, concert details, reviews and photos. Find information on all of Coldplay's upcoming concerts, tour dates and ticket information for 2025-2026. Unfortunately there are no concert dates for Coldplay ... The tour began at San José's Estadio Nacional de Costa Rica on 18 March 2022, paused at London's Wembley Stadium on 12 September 2025 and the band expect to ... Wembley and Hull dates announced (Only European shows). T*ckets go on s*le on September 27th (9am) · r/Coldplay - Wembley and Hull ... Dec 10, 2025 NEW COLDPLAY VR CONCERT, GAME PACK and WORKOUTS ANNOUNCED ; Nov 04, 2025 Win tickets to Jonny & Chris' London show ; Aug 30, 2025 7th and 8th ... Coldplay. There aren't any Coldplay events right now. Follow Coldplay on viagogo t

In [21]:
# ------------------------------------------------------------
# STATE DEL INVESTIGADOR
# Define los datos que viajan entre nodos
# ------------------------------------------------------------

class InvestigatorState(TypedDict):
    # Historial de mensajes para el LLM
    messages: Annotated[List[BaseMessage], add_messages]

    # Lista final de eventos (conciertos / festivales)
    events: List[Dict]

In [22]:
# ------------------------------------------------------------
# TOOLS OPCIONALES (el agente decide si usarlas o no)
# ------------------------------------------------------------

@tool
def search_events_tool(artist: str) -> List[Dict]:
    """
    Busca conciertos y festivales próximos de un artista usando Google Serper.
    Devuelve una lista inicial de eventos sin normalizar.
    """
    query = f"upcoming concerts and festivals {artist} Europe"
    results = serper.run(query)

    events = []
    for r in results.split("\n"):
        events.append({
            "id": hash(r),
            "artist": artist,
            "raw_text": r,
            "city": "Unknown",
            "country": "Unknown",
            "date": "Unknown",
            "price": "Unknown",
            "type": "concert",
            "link": "Unknown"
        })

    return events


@tool
def filter_by_country_tool(events: List[Dict], country: str) -> List[Dict]:
    """Filtra eventos por país."""
    return [e for e in events if country.lower() in e.get("raw_text", "").lower()]


@tool
def filter_by_city_tool(events: List[Dict], city: str) -> List[Dict]:
    """Filtra eventos por ciudad."""
    return [e for e in events if city.lower() in e.get("raw_text", "").lower()]


@tool
def filter_festivals_tool(events: List[Dict]) -> List[Dict]:
    """Devuelve solo festivales."""
    return [
        e for e in events
        if "festival" in e.get("raw_text", "").lower()
    ]


@tool
def limit_results_tool(events: List[Dict], limit: int) -> List[Dict]:
    """Limita el número de resultados."""
    return events[:limit]


In [23]:
# ------------------------------------------------------------
# LÓGICA INTERNA 
# Estas funciones para normalizar
# ------------------------------------------------------------

def remove_duplicates(events: List[Dict]) -> List[Dict]:
    # Elimina duplicados usando el id como clave
    return list({e["id"]: e for e in events}.values())


def sort_by_date(events: List[Dict]) -> List[Dict]:
    # Ordena por fecha (las no válidas quedan al final)
    return sorted(events, key=lambda e: e.get("date", "9999-99-99"))


def normalize_price(events: List[Dict]) -> List[Dict]:
    # Asegura formato de precio homogéneo
    for e in events:
        price = e.get("price", "N/A")
        if price != "N/A" and "€" not in price:
            e["price"] = f"{price}€"
    return events


def normalize_location(events: List[Dict]) -> List[Dict]:
    # Garantiza que ciudad y país existan
    for e in events:
        e["city"] = e.get("city") or "Por confirmar"
        e["country"] = e.get("country") or "Por confirmar"
    return events


def normalize_type(events: List[Dict]) -> List[Dict]:
    # Asegura tipo válido de evento
    for e in events:
        if e.get("type") not in ("concert", "festival"):
            e["type"] = "concert"
    return events


def normalize_events(events: List[Dict]) -> List[Dict]:
    # Pipeline completo de normalización
    events = remove_duplicates(events)
    events = sort_by_date(events)
    events = normalize_price(events)
    events = normalize_location(events)
    events = normalize_type(events)
    return events




In [24]:
# ------------------------------------------------------------
# NODO INVESTIGADOR (AGENTE CON LLM)
# Decide si usar tools o no
# ------------------------------------------------------------

def investigator_node(state: InvestigatorState) -> InvestigatorState:
    system_prompt = (
        "Eres un investigador de eventos musicales.\n"
        "Tu tarea es encontrar conciertos y festivales de un artista.\n"
        "Usa herramientas SOLO si el usuario lo solicita explícitamente "
        "(país, ciudad, festivales, límite de resultados).\n"
        "Si no hay filtros, devuelve todos los eventos encontrados."
        "No inventes datos. Devuelve información estructurada."
    )

    llm = ChatCohere(
        model="command-a-03-2025",
        temperature=0,
        preamble=system_prompt
    )

    llm_with_tools = llm.bind_tools([
        search_events_tool,
        filter_by_country_tool,
        filter_by_city_tool,
        filter_festivals_tool,
        limit_results_tool
    ])

    # llm
    response = llm_with_tools.invoke(state["messages"])

    # Lógica fija que siempre se ejecuta normalizacion
    events = state.get("events", [])
    if events:
        events = normalize_events(events)

    return {
        "messages": [response],
        "events": events
    }

In [25]:

# ------------------------------------------------------------
# CONSTRUCCIÓN DEL GRAFO
# ------------------------------------------------------------

# Crear el grafo indicando el State
workflow = StateGraph(InvestigatorState)

# Registrar nodos
workflow.add_node("investigador", investigator_node)

workflow.add_node(
    "tools",
    ToolNode([
        search_events_tool,
        filter_by_country_tool,
        filter_by_city_tool,
        filter_festivals_tool,
        limit_results_tool
    ])
)

# Punto de entrada
workflow.add_edge(START, "investigador")

# Decisión del agente: usar tool o terminar
workflow.add_conditional_edges(
    "investigador",
    tools_condition,
    ["tools", END]
)

# Tras ejecutar una tool, volver al agente
workflow.add_edge("tools", "investigador")

# Compilar el grafo
investigator_graph = workflow.compile()




In [ ]:
# ------------------------------------------------------------
# EJECUCIÓN DE PRUEBA
# ------------------------------------------------------------
result = investigator_graph.invoke({
    "messages": [
        # --------------Casos de prueba----------------------------------------
        # Conciertos y festivales de Coldplay en España, solo 3 =NO HAY ENVENTOA
        #Conciertos y festivales de Taylor Swift en Europa, solo 3=ENCUENTRA DATOS PERO NO FILTRA PUES NO HAY PAIS O CIUDAD EN EL QUERY
        # ------------------------------------------------------------
        HumanMessage(content="Conciertos y festivales de Taylor Swift en Europa, solo 3")
    ],
    "events": []
})

result


{'messages': [HumanMessage(content='Conciertos y festivales de Taylor Swift en Europa, solo 3', additional_kwargs={}, response_metadata={}, id='937d0bd7-c542-4acc-bdfa-d8e5cfdfe5c0'),
  AIMessage(content='Primero, buscaré los conciertos y festivales de Taylor Swift en Europa. Luego, limitaré los resultados a 3.', additional_kwargs={'id': '3dc74feb-57db-495f-844c-1c821632081e', 'finish_reason': 'TOOL_CALL', 'tool_plan': 'Primero, buscaré los conciertos y festivales de Taylor Swift en Europa. Luego, limitaré los resultados a 3.', 'tool_calls': [{'id': 'search_events_tool_wa95egn9ndpz', 'type': 'function', 'function': {'name': 'search_events_tool', 'arguments': '{"artist":"Taylor Swift"}'}}], 'token_count': {'input_tokens': 1826.0, 'output_tokens': 65.0}}, response_metadata={'id': '3dc74feb-57db-495f-844c-1c821632081e', 'finish_reason': 'TOOL_CALL', 'tool_plan': 'Primero, buscaré los conciertos y festivales de Taylor Swift en Europa. Luego, limitaré los resultados a 3.', 'tool_calls': [{'